In [1]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Create dataset and dataloader

- we use FashionMNIST (see https://en.wikipedia.org/wiki/MNIST_database)
- the dataset is downloaded (if not already present)
- first we compute mean and std of the data over train dataset
- then we create new instance where normalization is applied

In [2]:
temp_dataset = datasets.FashionMNIST(root='./data', train=True, transform=transforms.ToTensor(), download=True)
temp_loader = DataLoader(dataset=temp_dataset, batch_size=len(temp_dataset), shuffle=False)

# Compute mean and std
data_iter = iter(temp_loader)
images, _ = next(iter(data_iter))  # Get all images

mean, std = images.mean().item(), images.std().item()
print(f'Mean {mean}')
print(f'Mean {std}')

100%|██████████| 26.4M/26.4M [00:02<00:00, 12.4MB/s]


Extracting ./data\FashionMNIST\raw\train-images-idx3-ubyte.gz to ./data\FashionMNIST\raw



100%|██████████| 29.5k/29.5k [00:00<00:00, 414kB/s]


Extracting ./data\FashionMNIST\raw\train-labels-idx1-ubyte.gz to ./data\FashionMNIST\raw



100%|██████████| 4.42M/4.42M [00:00<00:00, 12.1MB/s]


Extracting ./data\FashionMNIST\raw\t10k-images-idx3-ubyte.gz to ./data\FashionMNIST\raw



100%|██████████| 5.15k/5.15k [00:00<?, ?B/s]

Extracting ./data\FashionMNIST\raw\t10k-labels-idx1-ubyte.gz to ./data\FashionMNIST\raw



Mean 0.28604060411453247
Mean 0.3530242443084717


In [11]:
# FashionMNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((mean,), (std,))
])

# znormalizujeme
train_dataset = datasets.FashionMNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = datasets.FashionMNIST(root='./data', train=False, transform=transform, download=True)

32

# Train model

- define a model as a MLP class (see below), define your own architecture
- use regularization, optimizer, loss,... of your choice and define hyperparameters so that you obtain high accuracy

In [14]:
# Hyperparameters (add more)
batch_size = 32
learning_rate = 0.001
epochs = 15
hidden_sizes = [128, 64] # jak bude vypadat architektura modelu

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [20]:
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

784

In [23]:
# MLP Model, toto jsme si psali úplně sami, takže absolutně nevím jestli to mám správně, vlastně zkopírované z minulého cvika, akorát opravený dimenze
class MLP(nn.Module):
    def __init__(self, input_dim=784, hidden_dims=[64, 32], output_dim=10): # prej ve FashionMNIST je prej input 28*28=784, output zase prej 10
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dims[0])
        self.bn1 = nn.BatchNorm1d(hidden_dims[0])
        self.activation1 = nn.ReLU() #aktivace

        self.fc2 = nn.Linear(hidden_dims[0], hidden_dims[1]) #zase lineární
        self.bn2 = nn.BatchNorm1d(hidden_dims[1])
        self.activation2 = nn.ReLU() #aktivace

        self.head = nn.Linear(hidden_dims[1], output_dim)

    def forward(self, x):
        x = x.view(x.size(0), -1)   # flatten z [batch, 1, 28, 28] na [batch, 784], toto je akorát přidáno oproti minulému cviku, potřebuju vektor z těch tenzorů
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.activation1(x)

        x = self.fc2(x)
        x = self.bn2(x)
        x = self.activation2(x)
        x = self.head(x)
        return x

model = MLP().to(device)

In [25]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

# Training loop
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(train_loader):.4f}")

# Evaluation
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")

Epoch [1/15], Loss: 0.5283
Epoch [2/15], Loss: 0.3764
Epoch [3/15], Loss: 0.3436
Epoch [4/15], Loss: 0.3242
Epoch [5/15], Loss: 0.3094
Epoch [6/15], Loss: 0.2968
Epoch [7/15], Loss: 0.2882
Epoch [8/15], Loss: 0.2816
Epoch [9/15], Loss: 0.2743
Epoch [10/15], Loss: 0.2679
Epoch [11/15], Loss: 0.2610
Epoch [12/15], Loss: 0.2573
Epoch [13/15], Loss: 0.2524
Epoch [14/15], Loss: 0.2498
Epoch [15/15], Loss: 0.2462
Test Accuracy: 88.40%
